### torch nn.Embedding

In [2]:
import nltk

nltk.download('punkt') # NLTK 토크나이저
nltk.download('punkt_tab') # punkt 관련 테이블 리소스
nltk.download('stopwords') # 불용어 목록

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

사전학습된 임베딩 사용하지 않는 경우

In [3]:
sentences = [          
    'nice great best amazing',  # 긍정 문장 예시
    'stop lies',                # 부정/비판 문장 예시
    'pitiful nerd',             # 부정 문장 예시
    'excellent work',           # 긍정 문장 예시
    'supreme quality',          # 긍정 문장 예시
    'bad',                      # 부정 문장 예시
    'highly respectable'        # 긍정 문장 예시
]                               # 분류 모델에 넣을 입력 문장 리스트(list[str])
labels = [1, 0, 0, 1, 1, 0, 1]  # 각 문장에 대한 이진 라벨(1=긍정, 0=부정)

In [4]:
# NLTK 토크나이저로 토큰화
from nltk.tokenize import word_tokenize      

tokenized_sentences = [word_tokenize(sent) for sent in sentences]      
tokenized_sentences

[['nice', 'great', 'best', 'amazing'],
 ['stop', 'lies'],
 ['pitiful', 'nerd'],
 ['excellent', 'work'],
 ['supreme', 'quality'],
 ['bad'],
 ['highly', 'respectable']]

In [5]:
# 단어 사전 생성 + 정수 인코딩
from collections import Counter

tokens = [token for sent in tokenized_sentences for token in sent]  # 문장 리스트를 1차원으로 평탄화
word_counts = Counter(tokens)  # 전체 토큰의 등장 갯수
print(word_counts)

word_to_index = {word: index + 2 for index, word in enumerate(tokens)}  # 토큰을 순서대로 인덱싱 (인덱스 +2)
word_to_index['<PAD>'] = 0  # 패딩토큰 추가
word_to_index['<UNK>'] = 1  # OOV 토큰 추가
word_to_index = dict(sorted(word_to_index.items(), key=lambda x:x[1]))  # 딕셔너리 정렬 (인덱스 순)
print(word_to_index)

vocab_size = len(word_to_index)  # 특수토큰 포함 전체 어휘 수
vocab_size  # 인덱스 기준으로 순차적으로 정렬된 모습

Counter({'nice': 1, 'great': 1, 'best': 1, 'amazing': 1, 'stop': 1, 'lies': 1, 'pitiful': 1, 'nerd': 1, 'excellent': 1, 'work': 1, 'supreme': 1, 'quality': 1, 'bad': 1, 'highly': 1, 'respectable': 1})
{'<PAD>': 0, '<UNK>': 1, 'nice': 2, 'great': 3, 'best': 4, 'amazing': 5, 'stop': 6, 'lies': 7, 'pitiful': 8, 'nerd': 9, 'excellent': 10, 'work': 11, 'supreme': 12, 'quality': 13, 'bad': 14, 'highly': 15, 'respectable': 16}


17

In [6]:
# 토큰화된 문장 리스트를 받아 단어 -> 인덱스 사전으로 정수 시퀀스 목록으로 2차원 형태로 만들어주는 함수
def text_to_sequences(sentences, word_to_index):
    sequences = []

    for sent in sentences:  # 문장 단위 순회
        sequence = []

        for token in sent:  # 토큰 단위 순회
            if token in word_to_index:  # 사전에 있는 단어면
                sequence.append(word_to_index[token])  # 해당 단어의 값(ID) 추가
            else:  # 사전에 없으면
                sequence.append(word_to_index['<UNK>'])  # 해당 위치에 OOV 토큰 추가
            
        sequences.append(sequence)

    return sequences

sequences = text_to_sequences(tokenized_sentences, word_to_index)
sequences

[[2, 3, 4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14], [15, 16]]

In [7]:
# 패딩 추가
import numpy as np

# 서로 다른 길이의 정수 시퀀스를 0(<PAD>)으로 채워 (문장수, maxlen) 형태로 맞추는 함수
def pad_sequences(sequences, maxlen):
    # (문장수 x maxlen) 크기의 0 패딩 생성
    padded_sequences = np.zeros((len(sequences), maxlen), dtype=int)
    for index, seq in enumerate(sequences):
        # index번째 행에서 0번 위치부터 len(seq)-1 위치까지를 seq의 처음부터 maxlen개까지 사용
        padded_sequences[index, :len(seq)] = seq[:maxlen]  # 앞에서부터 시퀀스 채움. 시퀀스가 길면 maxlen으로 자름
    return padded_sequences

padded_sequences = pad_sequences(sequences, maxlen=4)
padded_sequences  # (문장 수, maxlen=4) 형태의 정수 배열

array([[ 2,  3,  4,  5],
       [ 6,  7,  0,  0],
       [ 8,  9,  0,  0],
       [10, 11,  0,  0],
       [12, 13,  0,  0],
       [14,  0,  0,  0],
       [15, 16,  0,  0]])

In [8]:
padded_sequences.shape  # 문장 수, 고정길이 4

(7, 4)

In [9]:
import torch
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 점수 시퀀스를 임베딩 -> RNN -> 선형층으로 처리해서 이진분류 logit(1개) 출력하는 모델
class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        # 단어 ID를 밀집 벡터로 변환하는 임베딩
        self.embeding = nn.Embedding(
            num_embeddings=vocab_size,  # 단어 사전 크기
            embedding_dim=embedding_dim,  # 임베딩 차원
            padding_idx=0  # 패딩 0번 인덱스는 업데이트 하지 않음
        )
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)  # RNN 입력(배치, 길이, 차원)
        # 마지막 은닉 상태를 받아 1차원 logit값으로 변환 (차후 BECWithLogitsLoss 등 사용해서 확률값 변환해야)
        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        embedded = self.embeding(x)  # (batch, seq_len) -> (batch, seq_len, embedding_dim) 임베딩 하고 난 후엔 임베딩만큼? 차원이 늘어남
        out, h_n = self.rnn(embedded)  # out: logit값, h_n: (num_layers*directions, batch, hidden_size)
        out = self.out(h_n.squeeze(0))  # (batch, hidden_size) -> (batch, 1) 형태로 스퀴즈해서 보냄
        return out  # logit 값

embedding_dim = 100  # 단어 벡터 차원 크기
model = SimpleNet(vocab_size, embedding_dim, hidden_size=16)
model

SimpleNet(
  (embeding): Embedding(17, 100, padding_idx=0)
  (rnn): RNN(100, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [10]:
%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [11]:
from torchinfo import summary

summary(model) # 모델의 레이어 구성/파라미터 수 요약정보

Layer (type:depth-idx)                   Param #
SimpleNet                                --
├─Embedding: 1-1                         1,700
├─RNN: 1-2                               1,888
├─Linear: 1-3                            17
Total params: 3,605
Trainable params: 3,605
Non-trainable params: 0

In [12]:
# 임베딩 가중치 확인
import pandas as pd 

wv = model.embeding.weight.data  # 임베딩 층의 가중치 행렬 (단어 ID x 임베딩차원)
print(wv.shape)  # (vocab_size, embedding_dim)

vocab = word_to_index.keys()  # 인덱스만 가져옴
pd.DataFrame(wv, index=vocab)  # 인덱스 추가해서 데이터프레임으로 확인

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,0.629040,0.835913,-2.385701,0.102035,-1.586072,0.906797,0.533880,1.097284,1.099197,2.522136,...,1.258189,0.506915,-0.859769,0.535940,0.778347,-0.689865,0.661836,1.132648,1.353950,-0.372448
nice,-1.882316,0.835625,1.464429,-0.519164,0.252600,0.890297,1.365613,0.645128,-0.802243,-1.054703,...,0.956981,0.648471,1.861165,-1.395970,0.628121,-0.056264,0.791168,-1.320578,-0.857630,-0.900517
great,-0.844430,0.500831,-1.595187,-0.066976,-0.236696,0.609163,1.412830,-0.785475,0.266012,0.000611,...,-0.649245,-0.436798,0.817477,0.809279,-0.903980,-1.623405,0.059417,-1.563076,1.472493,0.648379
best,0.159217,-0.849544,1.552471,-0.352182,-1.044854,-1.475545,-0.301115,-0.945367,0.279050,-0.325573,...,0.479048,-0.369103,0.399368,0.649464,-1.106950,-1.009846,0.185586,0.403104,1.346088,0.207485
amazing,1.280491,0.632804,0.573325,-0.583515,0.539990,0.653675,1.236876,0.920033,0.494877,-0.370299,...,-0.577597,0.381552,-0.375371,-0.487326,-1.231191,0.438350,-1.974932,-0.732569,-2.138143,1.940409
stop,-0.319887,-0.415816,1.722145,0.114687,-0.491882,-0.304441,1.606016,-0.174908,-0.054848,2.053537,...,1.118896,-0.141763,0.259249,1.390779,0.716160,-0.293423,0.347728,-1.695554,-0.416557,-1.178225
lies,-0.374044,1.027922,0.182227,-1.899782,2.320695,0.381775,-1.494091,-0.879142,-1.042765,-1.132031,...,0.557661,0.406492,-0.728454,-0.629539,0.751158,-0.445335,-0.587463,0.181842,-0.010079,1.569314
pitiful,-1.869065,0.028122,0.220519,0.647197,0.742831,0.486322,-0.840516,0.266131,-0.199337,-0.046010,...,1.030606,0.398735,1.633605,-0.910140,0.202858,1.901083,0.129322,-0.415501,-1.100175,-1.211522
nerd,-0.638767,-2.320063,1.501724,1.818519,-0.751280,0.745489,0.972704,0.102940,2.339213,1.458796,...,1.790321,0.733917,-0.088235,0.728145,0.057799,1.217004,0.456257,-0.651001,1.230060,1.269196


In [13]:
# 모델학습 준비 : 텐서 변환, DataLoader 구성, 손실함수/최적화함수 설정
X = torch.tensor(padded_sequences, dtype=torch.long)  # 입력 시퀀스는 LongTensor로 변환(Embedding 입력)
y = torch.tensor(labels, dtype=torch.float).unsqueeze(1)  # 라벨은 FloatTensor로 변환. (N, ) -> (N, 1)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

criterion = nn.BCEWithLogitsLoss()  # 시그모이드를 포함한 손실함수
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [14]:
for epoch in range(20):
    epoch_loss = 0

    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()  # 이전 기울기 초기화
        output = model(x_batch)  # 순전파
        loss = criterion(output, y_batch)  # 손실 계산 
        loss.backward()  # 역전파 : 기울기 계산
        optimizer.step()  # 파라미터 업데이트

        epoch_loss += loss.item()  # 미니배치 손실은 float 형태로 누적

    print(f'epoch {epoch} Loss : {epoch_loss / len(dataloader)}')

epoch 0 Loss : 0.6908832341432571
epoch 1 Loss : 0.5443932600319386
epoch 2 Loss : 0.49866949021816254
epoch 3 Loss : 0.44963910803198814
epoch 4 Loss : 0.4190492704510689
epoch 5 Loss : 0.346510399132967
epoch 6 Loss : 0.2629213333129883
epoch 7 Loss : 0.1862723045051098
epoch 8 Loss : 0.14090854674577713
epoch 9 Loss : 0.09538835287094116
epoch 10 Loss : 0.06654233299195766
epoch 11 Loss : 0.049990191124379635
epoch 12 Loss : 0.04026972874999046
epoch 13 Loss : 0.03344067372381687
epoch 14 Loss : 0.02638161461800337
epoch 15 Loss : 0.02386976545676589
epoch 16 Loss : 0.020690327510237694
epoch 17 Loss : 0.018355122301727533
epoch 18 Loss : 0.016881723888218403
epoch 19 Loss : 0.015689780237153172


In [15]:
# 평가 및 예측
model.eval()  # 평가모드

with torch.no_grad():  # 기울기 계산 비활성화
    output = model(X)  # 순전파 (예측값 생성) - 실제로는 테스트데이터
    prob = torch.sigmoid(output)  # 0~1 사이 확률로 변환
    pred = (prob >= 0.5).int()  # 임계값 0.5 기준으로 이진분류 (0/1)

print(labels)
print(pred.squeeze().detach().numpy())  # 1차원 배열형태로 변환

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]


사전학습된 임베딩 모델을 사용

In [16]:
from gensim.models import KeyedVectors

# 사전학습된 Wod2Vec 로드
model_wv = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True)
model_wv.vectors.shape  # (어휘 수, 300)

(3000000, 300)

In [17]:
# 임베딩 매트릭스 초기화 후 사전학습 임베딩 차원으로 재구성
print(len(word_to_index))

embedding_matrix = np.zeros((len(word_to_index), model_wv.vectors.shape[1]))
embedding_matrix.shape

17


(17, 300)

In [18]:
# 단어가 사전학습 모델에 있으면 임베딩 벡터 반환, 없으면 None 반환
def get_word_embedding(word):
    if word in model_wv:
        return model_wv[word]
    else:
        return None

get_word_embedding('nerd').shape  # (300,) => 300개 벡터 차원을 받아옴

(300,)

In [19]:
# 학습된 임베딩 벡터를 가져와 복사
for word, index in word_to_index.items():
    if index >= 2:  # pad, oov 제외
        emb = get_word_embedding(word)
        if emb is not None:
            embedding_matrix[index] = emb  # 해당 단어 인덱스 위치에 사전학습 벡터를 복사

In [20]:
# 임베딩 매트릭스 확인
pd.DataFrame(embedding_matrix, index=word_to_index.keys())

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
nice,0.158203,0.105957,-0.189453,0.386719,0.083496,-0.267578,0.083496,0.113281,-0.104004,0.178711,...,-0.085449,0.189453,-0.146484,0.134766,-0.040771,0.032715,0.089355,-0.267578,0.008362,-0.213867
great,0.071777,0.208008,-0.028442,0.178711,0.132812,-0.099609,0.096191,-0.116699,-0.008545,0.148438,...,-0.011475,0.064453,-0.289062,-0.048096,-0.199219,-0.071289,0.064453,-0.167969,-0.020874,-0.142578
best,-0.126953,0.021973,0.287109,0.153320,0.127930,0.032715,-0.115723,-0.029541,0.153320,0.011292,...,0.006439,-0.033936,-0.166016,-0.016846,-0.048584,-0.022827,-0.152344,-0.101562,-0.090332,0.088379
amazing,0.073730,0.004059,-0.135742,0.022095,0.180664,-0.046631,0.224609,-0.229492,-0.040039,0.225586,...,0.018433,-0.021240,-0.250000,-0.020142,-0.310547,-0.207031,-0.006317,-0.141602,-0.150391,-0.137695
stop,-0.057861,0.013184,0.115234,0.069824,-0.306641,-0.044678,0.048584,0.152344,0.073242,-0.100098,...,0.100098,0.171875,-0.113281,0.064453,-0.115723,0.048096,-0.004822,0.086426,0.029907,0.007812
lies,0.149414,-0.012817,0.328125,0.025513,0.017334,0.190430,0.188477,-0.143555,-0.090820,0.206055,...,-0.308594,0.183594,-0.202148,0.031494,-0.164062,-0.201172,0.080078,-0.105469,0.149414,0.157227
pitiful,0.269531,0.253906,-0.020996,0.060303,-0.010925,0.217773,0.139648,-0.057617,0.312500,0.253906,...,-0.063477,0.132812,-0.094238,0.089355,-0.065430,-0.016235,-0.107910,-0.072266,-0.094238,0.028809
nerd,0.265625,-0.207031,-0.026611,0.419922,-0.208984,0.390625,0.164062,0.063965,0.149414,-0.017700,...,0.215820,0.125000,-0.227539,-0.310547,-0.112793,-0.096680,0.255859,0.124023,-0.030273,0.082031


In [21]:
# nn.parameter를 활용해 학습가능 파라미터를 나눔 (이론)
a = torch.tensor([1., 2., 3.], requires_grad=False)
print(a.requires_grad)

b = torch.tensor([1., 2., 3.], requires_grad=True)
print(b.requires_grad)

False
True


In [22]:
import torch
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 점수 시퀀스를 임베딩 -> RNN -> 선형층으로 처리해서 이진분류 logit(1개) 출력하는 모델
class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        # 단어 ID를 밀집 벡터로 변환하는 임베딩
        self.embeding = nn.Embedding(
            num_embeddings=vocab_size,  # 단어 사전 크기
            embedding_dim=embedding_dim,  # 임베딩 차원
            padding_idx=0  # 패딩 0번 인덱스는 업데이트 하지 않음
        )

        # 사전학습된 임베딩벡터로 초기화
        self.embeding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float))
        self.embeding.weight.requires_grad = False  # True면 파인튜닝(추가학습), False면 임베딩 고정

        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)  # RNN 입력(배치, 길이, 차원)
        # 마지막 은닉 상태를 받아 1차원 logit값으로 변환 (차후 BECWithLogitsLoss 등 사용해서 확률값 변환해야)
        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        embedded = self.embeding(x)  # (batch, seq_len) -> (batch, seq_len, embedding_dim) 임베딩 하고 난 후엔 임베딩만큼? 차원이 늘어남
        out, h_n = self.rnn(embedded)  # out: logit값, h_n: (num_layers*directions, batch, hidden_size)
        out = self.out(h_n.squeeze(0))  # (batch, hidden_size) -> (batch, 1) 형태로 스퀴즈해서 보냄
        return out  # logit 값

embedding_dim = 300  # 단어 벡터 차원 크기
model = SimpleNet(vocab_size, embedding_dim, hidden_size=16)
model

SimpleNet(
  (embeding): Embedding(17, 300, padding_idx=0)
  (rnn): RNN(300, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [23]:
# 모델학습 준비 : 텐서 변환, DataLoader 구성, 손실함수/최적화함수 설정
X = torch.tensor(padded_sequences, dtype=torch.long)  # 입력 시퀀스는 LongTensor로 변환(Embedding 입력)
y = torch.tensor(labels, dtype=torch.float).unsqueeze(1)  # 라벨은 FloatTensor로 변환. (N, ) -> (N, 1)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

criterion = nn.BCEWithLogitsLoss()  # 시그모이드를 포함한 손실함수
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [24]:
for epoch in range(20):
    epoch_loss = 0

    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()  # 이전 기울기 초기화
        output = model(x_batch)  # 순전파
        loss = criterion(output, y_batch)  # 손실 계산 
        loss.backward()  # 역전파 : 기울기 계산
        optimizer.step()  # 파라미터 업데이트

        epoch_loss += loss.item()  # 미니배치 손실은 float 형태로 누적

    print(f'epoch {epoch} Loss : {epoch_loss / len(dataloader)}')

epoch 0 Loss : 0.7077058702707291
epoch 1 Loss : 0.6453593224287033
epoch 2 Loss : 0.5810692459344864
epoch 3 Loss : 0.4968421682715416
epoch 4 Loss : 0.39937451481819153
epoch 5 Loss : 0.27992957830429077
epoch 6 Loss : 0.17392363585531712
epoch 7 Loss : 0.11375118792057037
epoch 8 Loss : 0.07413643971085548
epoch 9 Loss : 0.049167776480317116
epoch 10 Loss : 0.038504545111209154
epoch 11 Loss : 0.028646637219935656
epoch 12 Loss : 0.02397556835785508
epoch 13 Loss : 0.020049894228577614
epoch 14 Loss : 0.017501211492344737
epoch 15 Loss : 0.01500418153591454
epoch 16 Loss : 0.014192520175129175
epoch 17 Loss : 0.012905112002044916
epoch 18 Loss : 0.011441555572673678
epoch 19 Loss : 0.010876829968765378


In [25]:
# 평가 및 예측
model.eval()  # 평가모드

with torch.no_grad():  # 기울기 계산 비활성화
    output = model(X)  # 순전파 (예측값 생성) - 실제로는 테스트데이터
    prob = torch.sigmoid(output)  # 0~1 사이 확률로 변환
    pred = (prob >= 0.5).int()  # 임계값 0.5 기준으로 이진분류 (0/1)

print(labels)
print(pred.squeeze().detach().numpy())  # 1차원 배열형태로 변환

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]
